In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
#ingest comapny_metadata_json_file
df = spark.read.format('json')\
        .option('multiline', True)\
        .load('/Volumes/yipidata/source/source_data/company_metadata/company_metadata.json')

In [0]:
#get column list and number of columns avaialble
companies = df.columns
no_of_companies = len(companies)

In [0]:
no_of_companies

21

In [0]:
#iterate through the column list to get the label name and column names
company_list = [f"'{c}',`{c}`" for c in companies]
company_list

["'Airbnb',`Airbnb`",
 "'Amazon Web Services',`Amazon Web Services`",
 "'Anthropic',`Anthropic`",
 "'Cloudflare',`Cloudflare`",
 "'Confluent',`Confluent`",
 "'DataRobot',`DataRobot`",
 "'Databricks',`Databricks`",
 "'Elastic',`Elastic`",
 "'Google DeepMind',`Google DeepMind`",
 "'Meta AI',`Meta AI`",
 "'Microsoft',`Microsoft`",
 "'MongoDB',`MongoDB`",
 "'NVIDIA',`NVIDIA`",
 "'OpenAI',`OpenAI`",
 "'Palantir',`Palantir`",
 "'Scale AI',`Scale AI`",
 "'Snowflake',`Snowflake`",
 "'SpaceX',`SpaceX`",
 "'Stripe',`Stripe`",
 "'Tesla',`Tesla`",
 "'Uber',`Uber`"]

In [0]:
#convert to list structure
company_list = ', '.join(company_list)
company_list

"'Airbnb',`Airbnb`, 'Amazon Web Services',`Amazon Web Services`, 'Anthropic',`Anthropic`, 'Cloudflare',`Cloudflare`, 'Confluent',`Confluent`, 'DataRobot',`DataRobot`, 'Databricks',`Databricks`, 'Elastic',`Elastic`, 'Google DeepMind',`Google DeepMind`, 'Meta AI',`Meta AI`, 'Microsoft',`Microsoft`, 'MongoDB',`MongoDB`, 'NVIDIA',`NVIDIA`, 'OpenAI',`OpenAI`, 'Palantir',`Palantir`, 'Scale AI',`Scale AI`, 'Snowflake',`Snowflake`, 'SpaceX',`SpaceX`, 'Stripe',`Stripe`, 'Tesla',`Tesla`, 'Uber',`Uber`"

In [0]:
#create stack expression. n, key_n, value_n as new_col1, new_col2
company_stack_expr = f"stack({no_of_companies}, {company_list}) as (company, data)"

In [0]:
company_stack_expr

"stack(21, 'Airbnb',`Airbnb`, 'Amazon Web Services',`Amazon Web Services`, 'Anthropic',`Anthropic`, 'Cloudflare',`Cloudflare`, 'Confluent',`Confluent`, 'DataRobot',`DataRobot`, 'Databricks',`Databricks`, 'Elastic',`Elastic`, 'Google DeepMind',`Google DeepMind`, 'Meta AI',`Meta AI`, 'Microsoft',`Microsoft`, 'MongoDB',`MongoDB`, 'NVIDIA',`NVIDIA`, 'OpenAI',`OpenAI`, 'Palantir',`Palantir`, 'Scale AI',`Scale AI`, 'Snowflake',`Snowflake`, 'SpaceX',`SpaceX`, 'Stripe',`Stripe`, 'Tesla',`Tesla`, 'Uber',`Uber`) as (company, data)"

In [0]:
transposed_df = df.select(expr(company_stack_expr))

In [0]:
final_df = transposed_df.select('company', 'data.*')
final_df.display()

company,employee_count,founded_year,headquarters,industry,is_public,stock_ticker
Airbnb,19967,1999,"London, UK",Data Analytics,false,null
Amazon Web Services,27826,2015,"Berlin, Germany",SaaS,false,null
Anthropic,43747,2006,"San Francisco, CA",FinTech,false,null
Cloudflare,4422,1998,"Austin, TX",Cybersecurity,false,CLOU
Confluent,3884,1995,"London, UK",Cloud Computing,false,null
DataRobot,23471,2012,"New York, NY",Cloud Computing,true,null
Databricks,17962,2003,"Berlin, Germany",Data Analytics,true,DATA
Elastic,24508,2016,"San Francisco, CA",Cybersecurity,true,ELAS
Google DeepMind,24274,2017,"Austin, TX",Data Analytics,false,null
Meta AI,20240,2001,"Seattle, WA",Data Analytics,true,null


In [0]:
final_df.write.format('delta')\
    .mode('overwrite')\
    .saveAsTable('yipidata.bronze.company_metadata')

In [0]:
%sql
select * from yipidata.bronze.company_metadata

company,employee_count,founded_year,headquarters,industry,is_public,stock_ticker
Airbnb,19967,1999,"London, UK",Data Analytics,false,null
Amazon Web Services,27826,2015,"Berlin, Germany",SaaS,false,null
Anthropic,43747,2006,"San Francisco, CA",FinTech,false,null
Cloudflare,4422,1998,"Austin, TX",Cybersecurity,false,CLOU
Confluent,3884,1995,"London, UK",Cloud Computing,false,null
DataRobot,23471,2012,"New York, NY",Cloud Computing,true,null
Databricks,17962,2003,"Berlin, Germany",Data Analytics,true,DATA
Elastic,24508,2016,"San Francisco, CA",Cybersecurity,true,ELAS
Google DeepMind,24274,2017,"Austin, TX",Data Analytics,false,null
Meta AI,20240,2001,"Seattle, WA",Data Analytics,true,null
